# Merton calibration & Monte Carlo — period 2014

**Period file:** **2014-01-01 → 2014-12-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** choose lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM exercise decision on SPY American calls (risk-neutral paths from the same simulator); results in `stopping_results`.

True rolling rule: at each update, re-estimate \(\hat\mu,\hat\sigma,\hat\lambda,\hat\mu_J,\hat\kappa\) (and \(\hat\sigma_J\) for the jump-size law) from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.



## 0. Setup



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
PERIOD_START = pd.Timestamp("2014-01-01")
PERIOD_END = pd.Timestamp("2014-12-31")
TICKERS = ["AAPL", "MSFT", "SPY"]
N_DAYS = 252
N_STEPS = 500  # Monte Carlo time steps per path
JUMP_THRESH = 3.0  # flag |r| > c * daily σ as a jump
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "3 months": pd.DateOffset(months=3),
    "6 months": pd.DateOffset(months=6),
    "1 year": pd.DateOffset(years=1),
    "2 years": pd.DateOffset(years=2),
    "3 years": pd.DateOffset(years=3),
    "5 years": pd.DateOffset(years=5),
}
ROLLING_OPTIONS = ["daily", "monthly", "none"]

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

rolling = {}
cal_meta = {}

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()



## 1. Stock price trends (2014)

Adjusted close for AAPL, MSFT, and SPY (primary).



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, 2014")
axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period 2014", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))



## 2. Strike prices in this period

Unique strikes \(K\) from `*_options_panel.csv` with `trading_date` in **2014-01-01 → 2014-12-31**.

> AAPL strikes are on the option/contract scale; equity adj closes are split-adjusted.



In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)



## 3. Estimation formulas (Merton jump-diffusion)

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa)\, dt + \sigma\, dW_t + (e^J - 1)\, dN_t$$

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \lambda\kappa - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z + \sum_{i=1}^{N_{\Delta t}} J_i\Big)$$

with \(Z\sim N(0,1)\), \(N_{\Delta t}\sim\mathrm{Poisson}(\lambda\Delta t)\), \(J_i\sim N(\mu_J,\sigma_J^2)\).

| Parameter | Estimator (from daily log returns) |
|-----------|-------------------------------------|
| \(\hat\mu\) | \(\bar r_{\text{non-jump}} \times 252\) |
| \(\hat\sigma\) | \(s_{\text{non-jump}} \times \sqrt{252}\) |
| Jump days | \(|r_t| > c\cdot\hat\sigma_{\text{day}}\) with \(c=3\) |
| \(\hat\lambda\) | \(n_{\text{jumps}} / Y\) (jumps per year) |
| Jump size \(\hat\mu_J\) | mean of jump-day returns \(J\) |
| \(\hat\sigma_J\) | std of jump-day returns (needed for the jump-size law) |
| Compensation \(\kappa\) | \(e^{\hat\mu_J + \hat\sigma_J^2/2} - 1\) |

**Five calibrated quantities shown in §4:** \(\mu,\sigma,\lambda,\mu_J,\kappa\). \(\sigma_J\) is also rolled for simulation.

**True rolling:** at each update date, re-estimate from the lookback window ending there; those params drive the next Monte Carlo segment.

P from returns (3σ jumps). Jump-size premium \(\mu_J^*\) from listed calls (Pan 2002); \(\lambda^*=\lambda\) (timing not separately identified). Do not set \(P=Q\) for jumps when \(\lambda>0\). §5 uses \(P\); §6 LSM uses \(Q\).




## 4. Calibration only — 2014

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.



In [ ]:
import sys
from pathlib import Path as _PqPath
_PQ_SCRIPTS = str((_PqPath("..") / "scripts").resolve())
if _PQ_SCRIPTS not in sys.path:
    sys.path.insert(0, _PQ_SCRIPTS)
from heston_option_calibration import load_calls_panel, select_quotes_asof, quotes_fingerprint
from pq_risk_premium import (
    estimate_merton_p,
    calibrate_merton_q,
    report_merton_pq,
    simulate_merton_q,
)

def _heston_opt_dir():
    if "OPT_DIR" in globals():
        return Path(OPT_DIR)
    data = Path(DATA)
    if int(N_DAYS) > 400:
        return data / "options" / "processed" / "short_interval"
    return data / "options" / "processed"

def estimate_merton_params(log_rets: pd.Series, jump_thresh: float = JUMP_THRESH):
    """Estimate μ, σ, λ, μ_J, σ_J, κ from a lookback window of daily log returns.

    V2 fix — residual variance allocation for diffusion σ:
    1) Identify jumps with the same 3σ threshold (for λ, μ_J, σ_J, κ only).
    2) Let σ²_total = Var(all daily log returns) × 252.
    3) Jump contribution (compound-Poisson, annual): λ(μ_J² + σ_J²).
    4) σ²_diff = max(σ²_total − λ(μ_J² + σ_J²), ε), σ = √σ²_diff.

    This keeps jump variance in the jump block instead of deleting it from
    diffusion (V1 used non-jump days only for σ).
    """
    x = log_rets.dropna()
    n = int(x.shape[0])
    nan6 = (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, n)
    if n < 2:
        return nan6

    sigma_day = float(x.std(ddof=1))
    if not np.isfinite(sigma_day) or sigma_day <= 0:
        return nan6

    jump_mask = np.abs(x.values) > jump_thresh * sigma_day
    jumps = x.iloc[jump_mask]
    normal = x.iloc[~jump_mask]
    base = normal if int(normal.shape[0]) >= 2 else x

    # Drift still from non-jump days (P-measure; unused under RN μ→r)
    mu = float(base.mean() * N_DAYS)

    years = n / float(N_DAYS)
    n_jumps = int(jump_mask.sum())
    lam = float(n_jumps / years) if years > 0 else 0.0

    if n_jumps >= 2:
        mu_j = float(jumps.mean())
        sigma_j = float(jumps.std(ddof=1))
    elif n_jumps == 1:
        mu_j = float(jumps.iloc[0])
        sigma_j = 0.0
    else:
        mu_j = 0.0
        sigma_j = 0.0

    if not np.isfinite(sigma_j) or sigma_j < 0:
        sigma_j = 0.0

    kappa = float(np.exp(mu_j + 0.5 * sigma_j**2) - 1.0)

    # Full-sample annualized variance; allocate jump second moment to jumps
    var_total_ann = float(x.var(ddof=1) * N_DAYS)
    jump_var_ann = float(max(lam, 0.0) * (mu_j**2 + sigma_j**2))
    sigma2_diff = var_total_ann - jump_var_ann
    eps = 1e-12
    if not np.isfinite(sigma2_diff) or sigma2_diff < eps:
        sigma2_diff = max(var_total_ann, eps)
    sigma = float(np.sqrt(sigma2_diff))

    return mu, sigma, lam, mu_j, sigma_j, kappa, n

def _slice_window(rets: pd.Series, end: pd.Timestamp, offset: pd.DateOffset) -> pd.Series:
    start = end - offset
    return rets.loc[(rets.index > start) & (rets.index <= end)]


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    if rolling_mode == "daily":
        update_dates = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    elif rolling_mode == "monthly":
        t0 = period_prices[ticker].dropna().index[0]
        month_ends = pd.date_range(PERIOD_START, PERIOD_END, freq="ME")
        update_dates = pd.DatetimeIndex([t0]).append(month_ends).unique().sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_prices[ticker].dropna().index[0]])

    panel = load_calls_panel(Path(DATA), ticker, opt_dir=_heston_opt_dir())
    prev_q = None
    prev_fp = None
    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        quotes = select_quotes_asof(panel, pd.Timestamp(t_u), offset)
        fp = quotes_fingerprint(quotes)
        p = estimate_merton_p(window, N_DAYS, float(globals().get("JUMP_THRESH", 3.0)))
        if not p["success"] or p["n"] < 2 or not np.isfinite(p["mu"]):
            continue
        if prev_q is not None and fp == prev_fp and fp:
            q = prev_q
        else:
            q = calibrate_merton_q(p, quotes)
            if q["success"]:
                prev_q, prev_fp = q, fp
            elif prev_q is not None:
                q = prev_q
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min() if len(window) else pd.Timestamp(t_u),
            "window_end": window.index.max() if len(window) else pd.Timestamp(t_u),
            "n_days": p["n"],
            "mu": p["mu"],
            "sigma": p["sigma"],
            "lam": p["lam"],
            "mu_j": p["mu_j"],
            "sigma_j": p["sigma_j"],
            "kappa": p["kappa"],
            "mu_j_q": q.get("mu_j_q", np.nan),
            "kappa_q": q.get("kappa_q", np.nan),
            "n_quotes": q.get("n_quotes", 0),
            "q_success": bool(q.get("success", False)),
            "jump_premium_identified": bool(q.get("jump_premium_identified", False)),
        })

    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced trading-day grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]

def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()

    def _series(col):
        return cal[col].to_numpy(dtype=float)

    mu_arr = _series("mu")
    sig_arr = _series("sigma")
    lam_arr = _series("lam")
    muj_arr = _series("mu_j")
    sj_arr = _series("sigma_j")
    kap_arr = _series("kappa")

    mu_step = np.empty(n_steps, dtype=float)
    sig_step = np.empty(n_steps, dtype=float)
    lam_step = np.empty(n_steps, dtype=float)
    muj_step = np.empty(n_steps, dtype=float)
    sj_step = np.empty(n_steps, dtype=float)
    kap_step = np.empty(n_steps, dtype=float)

    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        mu_step[i] = mu_arr[idx]
        sig_step[i] = sig_arr[idx]
        lam_step[i] = lam_arr[idx]
        muj_step[i] = muj_arr[idx]
        sj_step[i] = sj_arr[idx]
        kap_step[i] = kap_arr[idx]

    return dates, mu_step, sig_step, lam_step, muj_step, sj_step, kap_step, float(hist.iloc[0]), hist


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Five rolling-parameter graphs: μ, σ, λ, μ_J, κ."""
    panels = [
        ("mu", "μ̂ (annual)", "Estimated drift"),
        ("sigma", "σ̂ (annual)", "Estimated diffusion volatility"),
        ("lam", "λ̂ (jumps/year)", "Estimated jump intensity"),
        ("mu_j", "μ̂_J (log jump)", "Estimated jump size (mean)"),
        ("kappa", "κ̂", "Jump compensation"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
        for ax, (col, ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = pd.to_datetime(r["date"])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[col], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            if col in {"mu", "mu_j", "kappa"}:
                ax.axhline(0, color="0.5", lw=0.8)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)


# Reset kernel-side UI handles so reopen + Run All cannot reuse stale widgets
plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 years",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="monthly",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (graphs only)</b> — lookback + rolling, then <b>Reestimate</b>. "
        "Shows μ̂, σ̂, λ̂, μ̂_J, κ̂. No Monte Carlo here."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated:** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)

        p_rows, prem_rows, q_rows = [], [], []
        for t in TICKERS:
            if rolling[t] is None or len(rolling[t]) == 0:
                continue
            last = rolling[t].iloc[-1]
            p_tbl, prem, q_tbl = report_merton_pq(last)
            p_tbl["ticker"] = t
            prem["ticker"] = t
            q_tbl["ticker"] = t
            p_rows.append(p_tbl)
            prem_rows.append(prem)
            q_rows.append(q_tbl)
        if p_rows:
            display(Markdown("### Physical-measure (P) parameters"))
            display(pd.DataFrame(p_rows))
            display(Markdown("### Jump-size risk premium (Pan $\\mu-\\mu^*$, from listed calls)"))
            display(pd.DataFrame(prem_rows))
            display(Markdown("### Risk-neutral (Q) dynamics"))
            display(pd.DataFrame(q_rows))
        display(Markdown(
            "Go to **§5** and click **Start** for one MC pair per company. "
            "Jump-size vol \(\sigma_J\) is estimated in the same windows and used in simulation."
        ))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()



## 5. Monte Carlo only — one graph pair per company (2014)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.



In [ ]:
def simulate_merton_rolling(
    mu_step, sigma_step, lam_step, muj_step, sj_step, kap_step, S0, n_paths, seed
):
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    for i in range(n_steps):
        mu = mu_step[i]
        sigma = sigma_step[i]
        lam = lam_step[i]
        mu_j = muj_step[i]
        sigma_j = sj_step[i]
        kappa = kap_step[i]
        z = rng.standard_normal(n_paths)
        n_jumps = rng.poisson(max(lam, 0.0) * dt, size=n_paths)
        jump_sizes = np.zeros(n_paths, dtype=float)
        mask = n_jumps > 0
        if mask.any():
            jump_sizes[mask] = (
                n_jumps[mask] * mu_j
                + np.sqrt(n_jumps[mask]) * max(sigma_j, 0.0) * rng.standard_normal(int(mask.sum()))
            )
        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * sigma**2 - lam * kappa) * dt
            + sigma * np.sqrt(dt) * z
            + jump_sizes
        )
    return paths


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        (
            dates_now, mu_now, sig_now, lam_now, muj_now, sj_now, kap_now, S0_now, hist_now
        ) = param_schedule_for_steps(ticker, rolling[ticker])
        paths = simulate_merton_rolling(
            mu_now, sig_now, lam_now, muj_now, sj_now, kap_now, S0_now, n_paths, seed
        )
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            x = dates_now[:_n]
            axes[0].plot(x, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Merton Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(x, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(x, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("date")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(100.0 * np.sqrt(np.mean(((p50 - _hist) / np.maximum(np.abs(_hist), 1e-8)) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · "
            f"**ICP (25–75)** = `{100*icp:.1f}%` · "
            f"**avg band width** = `{abw:.4f}` · "
            f"RMSE%(p50) = `{rmse:.2f}%` | seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)



## 6. Optimal stopping (American calls — Merton)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Merton simulator** and §4 calibration on **SPY / AAPL / MSFT** to decide exercise vs wait for American calls.

**Do not average simulated stock paths before stopping decisions.** Generate a cloud of individual risk-neutral paths (each keeps its own shocks). Then Longstaff–Schwartz:

1. At each exercise date, compute the immediate payoff $\max(S_t-K,0)$ **on every path**.
2. Estimate continuation by regression on in-the-money simulated states ($1, S, S^2$, path vol proxy).
3. **Each path** exercises iff payoff $>$ continuation; otherwise it continues.
4. Discount that path's stopping payoff to $t=0$.
5. The American value is the **average of those discounted payoffs** (then $\max$ with the $t=0$ intrinsic).

Paths for pricing are **$Q$** (Pan jump-size and/or volatility premia). §5 is $P$ visualization only. §5's expected-vs-history plot is visualization only — it is not the input to LSM.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping** (all three underlyings).




In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from pq_risk_premium import simulate_merton_q, report_merton_pq
from american_lsm import (
    STOP_TICKERS,
    lsm_american_call,
    load_calls,
    params_asof,
    sample_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Pan Q-Merton paths (λ* = λ, μ_J* from options)."""
    ticker = str(getattr(row, "underlying", "SPY")).upper()
    if ticker not in rolling or len(rolling[ticker]) == 0:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    p = params_asof(rolling[ticker], row.trading_date)
    if p is None or not np.isfinite(p.get("mu_j_q", np.nan)):
        raise RuntimeError(f"No Q jump premium for {ticker} — need option quotes in §4.")
    dte = int(row.dte)
    if dte < 2:
        raise ValueError("dte must be >= 2")
    r = float(row.r)
    S0 = float(row.S_t)
    n = dte
    steps = {
        "rf": np.full(n, r, dtype=float),
        "sigma": np.full(n, float(p["sigma"]), dtype=float),
        "lam": np.full(n, float(p["lam"]), dtype=float),
        "mu_j": np.full(n, float(p["mu_j"]), dtype=float),
        "mu_j_q": np.full(n, float(p["mu_j_q"]), dtype=float),
        "sigma_j": np.full(n, float(p["sigma_j"]), dtype=float),
        "kappa": np.full(n, float(p["kappa"]), dtype=float),
        "kappa_q": np.full(n, float(p["kappa_q"]), dtype=float),
    }
    return simulate_merton_q(steps, S0, n_paths, seed, n_days=int(N_DAYS) if int(N_DAYS) <= 400 else 252)



_STOP_TICKERS = list(STOP_TICKERS)
_contracts_by_ticker = {}
for _t in _STOP_TICKERS:
    _panel = load_calls(DATA, _t)
    _contracts_by_ticker[_t] = sample_calls(
        _panel, PERIOD_START, PERIOD_END, 
    )

stopping_results = {}  # ticker -> DataFrame

for _t in _STOP_TICKERS:
    _n = len(_contracts_by_ticker[_t])
    display(Markdown(
        f"Sampled **{_n}** {_t} American calls in "
        f"{PERIOD_START.date()} → {PERIOD_END.date()} "
        f"(one nearest-ATM call each Monday, or the next session if Monday is closed; DTE 7–60)."
    ))
    if _n:
        display(
            _contracts_by_ticker[_t][
                ["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]
            ].head(8)
        )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            missing = [t for t in _STOP_TICKERS if t not in rolling or len(rolling[t]) == 0]
            if missing:
                display(Markdown(
                    "Run **Reestimate** in §4 first "
                    f"(need calibration for: {', '.join(missing)})."
                ))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            dt = 1.0 / 252.0  # trading-day clock for LSM; never the 1-min N_DAYS
            stopping_results = {}

            for ticker in _STOP_TICKERS:
                contracts = _contracts_by_ticker[ticker]
                if contracts is None or len(contracts) == 0:
                    display(Markdown(f"No {ticker} call contracts in this period panel slice."))
                    continue

                rows = []
                example = None
                for i, row in enumerate(contracts.itertuples(index=False)):
                    paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                    res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                    err = res.price - float(row.option_price)
                    rows.append({
                        "ticker": ticker,
                        "trading_date": row.trading_date,
                        "S_t": float(row.S_t),
                        "K": float(row.K),
                        "dte": int(row.dte),
                        "r": float(row.r),
                        "market": float(row.option_price),
                        "model_price": res.price,
                        "error": err,
                        "early_ex_frac": res.early_exercise_frac,
                        "mean_ex_day": res.mean_exercise_step,
                    })
                    if example is None:
                        example = (row, paths, res)

                df = pd.DataFrame(rows)
                stopping_results[ticker] = df
                rmse = float(100.0 * np.sqrt(np.mean((df["error"] / np.maximum(np.abs(df["market"]), 1e-8)) ** 2)))
                mae = float(np.mean(np.abs(df["error"])))
                color = COLORS.get(ticker, "#2ca02c")

                display(Markdown(
                    f"### Merton — LSM results ({ticker})\n"
                    f"n_paths={n_paths} | contracts={len(df)} | "
                    f"RMSE={rmse:.2f}% | MAE={mae:.4f} | "
                    f"mean early-exercise fraction="
                    f"{df['early_ex_frac'].mean():.3f}"
                ))
                display(
                    df[
                        ["trading_date", "S_t", "K", "dte", "market", "model_price",
                         "error", "early_ex_frac", "mean_ex_day"]
                    ].round(4)
                )

                with plt.ioff():
                    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                    ax = axes[0]
                    ax.scatter(df["market"], df["model_price"], alpha=0.75, color=color)
                    lo = min(df["market"].min(), df["model_price"].min())
                    hi = max(df["market"].max(), df["model_price"].max())
                    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                    ax.set_xlabel("market option_price")
                    ax.set_ylabel("model LSM price")
                    ax.set_title("Price: model vs market")

                    axes[1].bar(
                        ["model", "market"],
                        [df["model_price"].mean(), df["market"].mean()],
                        color=[color, "#7f7f7f"],
                    )
                    axes[1].set_title("Mean option value")
                    axes[1].set_ylabel("price")

                    axes[2].hist(
                        df["mean_ex_day"], bins=12,
                        color=color, alpha=0.85, edgecolor="white",
                    )
                    axes[2].set_xlabel("mean exercise day (by contract)")
                    axes[2].set_title("Optimal exercise timing")
                    fig.suptitle(
                        f"Merton optimal stopping | {ticker} | "
                        f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                        fontsize=11, y=1.02,
                    )
                    fig.tight_layout()
                _show_fig(fig)

                if example is not None:
                    row, paths, res = example
                    j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                    t_ex = int(res.exercise_steps[j])
                    with plt.ioff():
                        fig2, ax = plt.subplots(figsize=(10, 3.8))
                        ax.plot(paths[j], color=color, lw=1.5, label="one RN path")
                        ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                        ax.scatter(
                            [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                            label=f"exercise day {t_ex}",
                        )
                        ax.set_xlabel("day")
                        ax.set_ylabel("S")
                        ax.set_title(
                            f"{ticker} example path | "
                            f"trade {pd.Timestamp(row.trading_date).date()} | "
                            f"dte={int(row.dte)} | model={res.price:.3f} vs "
                            f"mkt={float(row.option_price):.3f}"
                        )
                        ax.legend(frameon=False, loc="best")
                        fig2.tight_layout()
                    _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(dict keyed by ticker → model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY / AAPL / MSFT American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))





## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY / AAPL / MSFT calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.


